# Inspect Point Source Calibration Results

This notebook loads and inspects the pickled results from `PointSourceCalibrationPlugin`.

The plugin saves `model_components` containing:
- Gain solutions per receiver
- Temperature model components (atmospheric, point source, receiver, spillover)
- Beam gains for both polarizations

In [ ]:
import gc
import pickle

import matplotlib.pyplot as plt
import numpy as np
import numpy.ma as ma

# Set up plotting
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

In [ ]:
from scipy.ndimage import gaussian_filter1d, uniform_filter1d


def smooth_freq(spec, scale, method='gaussian'):
    """
    Mask-aware frequency smoothing along axis 0, preserving the original flags.
    spec   : (n_freq, n_recv) masked array
    scale  : boxcar window size in channels  (method='boxcar')
             OR Gaussian sigma in channels    (method='gaussian')
             boxcar_window ≈ 2.355 × gaussian_sigma (FWHM), e.g. sigma=9 ≈ a ~21-channel boxcar
             boxcar channels in the smoothing window should be odd (~0.13 MHz/chan in U-band)
    method : 'gaussian' or 'boxcar'
    """
    data  = np.ma.filled(spec, 0.0)
    valid = (~np.ma.getmaskarray(spec)).astype(float)
    if method == 'boxcar':
        num = uniform_filter1d(data,  size=int(scale), axis=0, mode='nearest')
        den = uniform_filter1d(valid, size=int(scale), axis=0, mode='nearest')
    elif method == 'gaussian':
        num = gaussian_filter1d(data,  sigma=scale, axis=0, mode='nearest')
        den = gaussian_filter1d(valid, sigma=scale, axis=0, mode='nearest')
    else:
        raise ValueError(f"method must be 'gaussian' or 'boxcar', got {method!r}")
    with np.errstate(invalid='ignore', divide='ignore'):
        out = num / den
    # preserve original flags (don't fill them in); den==0 guards fully-flagged windows
    mask = np.ma.getmaskarray(spec) | (den == 0)
    return np.ma.array(out, mask=mask)



## Load the pickle file

Update the path below to point to your actual results directory.

In [ ]:
# Update this path to your actual results directory
pickle_path = '/home/mgrsantos/projects/data/context/1675021905/point_source_calibration_plugin.pickle'

# Load the pickle file
with open(pickle_path, 'rb') as f:
    context = pickle.load(f)

print(f"Loaded context with keys: {list(context.keys())}")

## Access model_components

The `model_components` is stored as a result using `ResultEnum.MODEL_COMPONENTS`.

In [ ]:
# Access model_components from context using ResultEnum
from museek.enums.result_enum import ResultEnum

print(context.keys())
track_data = context.get(ResultEnum.TRACK_DATA).result
model_components = context.get(ResultEnum.MODEL_COMPONENTS).result

del context
gc.collect()

In [ ]:
# Filter out non-period metadata keys (e.g. 'receivers' added for ReadCalibratorGainsPlugin)
period_keys = [k for k in model_components if isinstance(model_components[k], dict)]

## Inspect structure for one period

In [ ]:
# Get first period
period = list(model_components.keys())[0]
period_data = model_components[period]

print(f"\nPeriod: {period}")
print(f"Calibrator: {period_data['calibrator']}")
print(f"Dump indices: {period_data['dump_indices'][:5]}... (showing first 5)")
print(f"\nTop-level keys: {list(period_data.keys())}")
print(f"\nTemperature components: {list(period_data['temperatures'].keys())}")

## Check array shapes

In [ ]:
print(f"\nArray shapes for period '{period}':")
print("\nReceiver-specific (with receiver dimension):")
print(f"  gain: {period_data['gain'].shape} -> (n_freq, n_receivers)")
print(f"  gain: {period_data['gain'].shape} -> (n_freq, n_receivers)")
print(f"  atmospheric: {period_data['temperatures']['atmospheric'].shape} -> (n_dumps, n_freq, n_receivers)")
print(f"  receiver: {period_data['temperatures']['receiver'].shape} -> (n_freq, n_receivers)")

print("\nReceiver-independent (no receiver dimension):")
print(f"  beam_gain_HH: {period_data['beam_gain_HH'].shape} -> (n_dumps, n_freq)")
print(f"  beam_gain_VV: {period_data['beam_gain_VV'].shape} -> (n_dumps, n_freq)")
print(f"  point_source_HH: {period_data['temperatures']['point_source_HH'].shape} -> (n_dumps, n_freq)")
print(f"  point_source_VV: {period_data['temperatures']['point_source_VV'].shape} -> (n_dumps, n_freq)")
print(f"  spillover_HH: {period_data['temperatures']['spillover_HH'].shape} -> (n_dumps, n_freq)")
print(f"  spillover_VV: {period_data['temperatures']['spillover_VV'].shape} -> (n_dumps, n_freq)")
print(f"  synchrotron: {period_data['temperatures']['synchrotron'].shape} -> (n_dumps, n_freq)")

n_dumps, n_freq = period_data['beam_gain_HH'].shape
n_receivers = period_data['gain'].shape[1]

print("\nDimensions:")
print(f"  n_dumps: {n_dumps}")
print(f"  n_freq: {n_freq}")
print(f"  n_receivers: {n_receivers}")

## Waterfall plot of raw visibilities

In [ ]:
i_receiver = 0
period = 'before_scan'  # or 'after_scan'

freq_MHz = track_data.frequencies.squeeze / 1.0e6
times = track_data.timestamps.squeeze
dumps = np.array(track_data._dumps())
flags = track_data.flags.combine(threshold=1)

dump_indices = model_components[period]['dump_indices']
select = np.isin(dumps, dump_indices)
times_period = times[select]

vis = track_data.visibility.squeeze[select, :, i_receiver]
flag = flags.squeeze[select, :, i_receiver]
vis_masked = np.ma.masked_array(vis, mask=flag)
#vis_masked = np.ma.masked_array(vis, mask=[0])
#vis_masked = vis_masked/ma.median(vis_masked, axis=1)[:, np.newaxis]
#vis_masked = vis_masked/ma.median(vis_masked, axis=0)

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.pcolormesh(freq_MHz, times_period - times_period[0], vis_masked, shading='auto', cmap='viridis')
plt.colorbar(im, ax=ax, label='Visibility [counts]')
ax.set_xlabel('Frequency [MHz]')
ax.set_ylabel('Time [s]')
ax.set_title(f'Raw visibilities — {period} ({model_components[period]["calibrator"]}) — receiver {track_data.receivers[i_receiver]}')
plt.tight_layout()
plt.show()

In [ ]:
# Elevation vs time of the calibrator tracks, per period (independent y-axes).
dumps = np.asarray(track_data._dumps())
times = track_data.timestamps.squeeze
el    = track_data.elevation.squeeze          # (n_time, n_antennas), degrees

fig, axes = plt.subplots(1, len(period_keys), figsize=(7 * len(period_keys), 4),
                         squeeze=False, sharey=False)   # <-- independent y-axes

for ax, period in zip(axes[0], period_keys):
    sel   = np.isin(dumps, np.asarray(model_components[period]['dump_indices']))
    t_min = (times[sel] - times[sel].min()) / 60.0
    ax.plot(t_min, np.median(el[sel], axis=1), '.-', ms=3, lw=0.6)   # median over dishes
    ax.set_title(f"{period} ({model_components[period]['calibrator']})")
    ax.set_xlabel('Time [min since period start]')
    ax.set_ylabel('Elevation [deg]')   # <-- per-panel label now that ranges differ
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()



## Plot gain solutions

In [ ]:
# Get frequency axis (need to get from context or calculate)

freq_MHz = track_data.frequencies.squeeze/1.0e6
freq_mask = (freq_MHz >= 580.0) & (freq_MHz <= 1015.0)

# Plot gain for some receivers
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for period_name in period_keys:
    period_data = model_components[period_name]
    gains = period_data['gain']  # (n_freq, n_receivers)
    calibrator = period_data['calibrator']

    # Plot first 10 receivers for clarity
    ax = axes[0] if 'before' in period_name else axes[1]

    for i_recv in range(6):
        ax.plot(freq_MHz[freq_mask], gains[freq_mask, i_recv], alpha=0.7, label=f'Recv {i_recv}' if period_name == list(model_components.keys())[0] else None)

    ax.set_xlabel('Frequency [MHz]')
    ax.set_ylabel('Gain')
    ax.set_title(f'{period_name}: {calibrator}')
    ax.grid(True, alpha=0.3)

axes[0].legend(loc='best', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
freq_mask = (freq_MHz >= 580.0) & (freq_MHz <= 1015.0)

before = model_components['before_scan']['gain']  # (n_freq, n_receivers)
after  = model_components['after_scan']['gain']   # (n_freq, n_receivers)

ratio = after / before  # (n_freq, n_receivers)

for pol, pol_char in [('HH', 'h'), ('VV', 'v')]:
    fig, ax = plt.subplots(figsize=(12, 5))
    for i_receiver, receiver in enumerate(track_data.receivers):
        if receiver.polarisation != pol_char:
            continue
        ax.plot(freq_MHz[freq_mask], ratio[freq_mask, i_receiver], label=str(receiver), alpha=0.8)

    ax.axhline(1, color='k', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Frequency [MHz]')
    ax.set_ylabel('gain ratio (after / before)')
    ax.set_title(f'Gain ratio after / before scan — {pol}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()



In [ ]:
freq_mask = (freq_MHz >= 580.0) & (freq_MHz <= 1015.0)

before = model_components['before_scan']['gain']  # (n_freq, n_receivers)
after  = model_components['after_scan']['gain']   # (n_freq, n_receivers)

ratio = after / before  # (n_freq, n_receivers)
percentage_fluctuations = 100*(ratio - ma.median(ratio,axis=0))/ma.median(ratio,axis=0)
for pol, pol_char in [('HH', 'h'), ('VV', 'v')]:
    fig, ax = plt.subplots(figsize=(12, 5))
    for i_receiver, receiver in enumerate(track_data.receivers):
        if receiver.polarisation != pol_char:
            continue
        ax.plot(freq_MHz[freq_mask], percentage_fluctuations[freq_mask, i_receiver], label=str(receiver), alpha=0.8)
        print(ma.median(percentage_fluctuations[freq_mask, i_receiver]))

    ax.axhline(0, color='k', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Frequency [MHz]')
    ax.set_ylabel('Percentage')
    ax.set_title(f'Gain ratio (after/before) variation — {pol}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Plot temperature components for first receiver

In [ ]:
i_recv = 0  # First receiver
polarisation = track_data.receivers[i_recv].polarisation
pol = "HH" if polarisation == "h" else "VV"
freq_mask = (freq_MHz >= 580.0) & (freq_MHz <= 1015.0)

periods = period_keys
dumps_arr = np.array(track_data._dumps())

# --- Figure 1: Waterfall plots for the four temperature components ---
component_labels = ["Atmospheric", f"Point Source ({pol})", f"Spillover ({pol})", "Synchrotron"]

fig, axes = plt.subplots(len(periods), 4, figsize=(24, 5 * len(periods)))
if len(periods) == 1:
    axes = axes[np.newaxis, :]

for j, period in enumerate(periods):
    period_data = model_components[period]
    calibrator = period_data["calibrator"]
    dump_indices = period_data["dump_indices"]
    select = np.isin(dumps_arr, dump_indices)
    times_period = track_data.timestamps.squeeze[select]
    times_min = (times_period - times_period[0]) / 60.0

    atm = period_data["temperatures"]["atmospheric"][:, :, i_recv]
    point_source = period_data["temperatures"][f"point_source_{pol}"]
    spillover = period_data["temperatures"][f"spillover_{pol}"]
    synchrotron = period_data["temperatures"]["synchrotron"]

    for k, (comp, label) in enumerate(zip([atm, point_source, spillover, synchrotron], component_labels)):
        ax = axes[j, k]
        im = ax.pcolormesh(freq_MHz, times_min, comp, shading="auto", cmap="viridis")
        plt.colorbar(im, ax=ax, label="Temperature [K]")
        ax.set_xlabel("Frequency [MHz]")
        ax.set_ylabel("Time [min]")
        ax.set_title(f"{period} — {calibrator}\n{label}")

plt.suptitle(f"Temperature Waterfalls — receiver {track_data.receivers[i_recv]}", fontsize=14)
plt.tight_layout()
plt.show()

# --- Figure 2: Median over time spectrum for all components ---
fig, axes = plt.subplots(1, len(periods), figsize=(8 * len(periods), 5))
if len(periods) == 1:
    axes = [axes]

for j, period in enumerate(periods):
    period_data = model_components[period]
    calibrator = period_data["calibrator"]

    atm = period_data["temperatures"]["atmospheric"][:, :, i_recv]
    point_source = period_data["temperatures"][f"point_source_{pol}"]
    spillover = period_data["temperatures"][f"spillover_{pol}"]
    synchrotron = period_data["temperatures"]["synchrotron"]
    receiver_temp = period_data["temperatures"]["receiver"][:, i_recv]
    model_total = atm + point_source + receiver_temp + spillover + synchrotron

    ax = axes[j]
    ax.plot(freq_MHz, np.median(atm, axis=0), label="Atmospheric", linewidth=1.5)
    ax.plot(freq_MHz, np.median(point_source, axis=0), label=f"Point Source ({pol})", linewidth=1.5)
    ax.plot(freq_MHz, np.median(spillover, axis=0), label=f"Spillover ({pol})", linewidth=1.5)
    ax.plot(freq_MHz, np.median(synchrotron, axis=0), label="Synchrotron", linewidth=1.5)
    ax.plot(freq_MHz, receiver_temp, label="Receiver", linewidth=1.5)
    ax.plot(freq_MHz, np.median(model_total, axis=0)+2.7255, label="Total (with CMB)", linewidth=2, linestyle='--', color='k')
    ax.set_xlabel("Frequency [MHz]")
    ax.set_ylabel("Temperature [K]")
    ax.set_title(f"{period} — {calibrator}\nMedian over time")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(f"Temperature Components (median over time) — receiver {track_data.receivers[i_recv]}", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
target_freq_MHz = 700.0

# Use the first available receiver
i_recv = 0
target_recv = str(track_data.receivers[i_recv])
pol = 'HH' if track_data.receivers[i_recv].polarisation == 'h' else 'VV'

# Find nearest frequency index
i_freq = np.argmin(np.abs(freq_MHz - target_freq_MHz))
actual_freq = freq_MHz[i_freq]

dumps_arr = np.array(track_data._dumps())

for period in period_keys:
    period_data = model_components[period]
    dump_indices = period_data['dump_indices']
    select = np.isin(dumps_arr, dump_indices)
    times = track_data.timestamps.squeeze[select]
    times_min = (times - times[0]) / 60.0
    on_mask = period_data['on_mask']

    atm       = period_data['temperatures']['atmospheric'][:, i_freq, i_recv]       # (n_dumps,)
    spillover  = period_data['temperatures'][f'spillover_{pol}'][:, i_freq]          # (n_dumps,)
    rec_temp   = period_data['temperatures']['receiver'][i_freq, i_recv]             # scalar
    synch      = period_data['temperatures']['synchrotron'][:, i_freq]               # (n_dumps,)
    point_src  = period_data['temperatures'][f'point_source_{pol}'][:, i_freq]       # (n_dumps,)

    fig, ax = plt.subplots(figsize=(10, 5))

    ax.plot(times_min, atm,                               label='Atmospheric',         linewidth=1.5)
    ax.plot(times_min, spillover,                         label=f'Spillover ({pol})',  linewidth=1.5)
    ax.plot(times_min, synch,                             label='Synchrotron',         linewidth=1.5)
    ax.plot(times_min, point_src,                         label=f'Point source ({pol})', linewidth=1.5)
    ax.axhline(rec_temp,                                  label='Receiver (static)',   linewidth=1.5, linestyle='--', color='k')

    # Mark on/off regions
    ax.fill_between(times_min, ax.get_ylim()[0], ax.get_ylim()[1],
                    where=on_mask, alpha=0.1, color='green', label='On source')

    ax.set_xlabel('Time [min]')
    ax.set_ylabel('Temperature [K]')
    ax.set_title(f'{period} ({period_data["calibrator"]}) — {target_recv} — {actual_freq:.1f} MHz')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
periods = period_keys
dumps_arr = np.array(track_data._dumps())
freq_mask = (freq_MHz >= 580.0) & (freq_MHz <= 1015.0)

for pol, pol_char in [('HH', 'h'), ('VV', 'v')]:
    fig, ax = plt.subplots(figsize=(10, 5))

    for i_recv, receiver in enumerate(track_data.receivers):
        if receiver.polarisation != pol_char:
            continue

        period_medians = []
        for period in periods:
            period_data = model_components[period]

            atm = period_data["temperatures"]["atmospheric"][:, :, i_recv]
            spillover = period_data["temperatures"][f"spillover_{pol}"]
            receiver_temp = period_data["temperatures"]["receiver"][:, i_recv]

            combined = atm + spillover + receiver_temp
            period_medians.append(np.median(combined[:, freq_mask], axis=0))

        mean_over_periods = np.mean(period_medians, axis=0)
        ax.plot(freq_MHz[freq_mask], mean_over_periods, label=str(receiver), linewidth=1.5, alpha=0.8)

    ax.set_xlabel("Frequency [MHz]")
    ax.set_ylabel("Temperature [K]")
    ax.set_title(f"Receiver + Spillover + Atmosphere — {pol}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()



## Average(on) - average(off) for model components (used in gain)

In [ ]:
freq_mask = (freq_MHz >= 580.0) & (freq_MHz <= 1015.0)
periods = period_keys
polarisations = ['HH', 'VV']

dumps = np.array(track_data._dumps())
flags = track_data.flags.combine(threshold=1)

fig, axes = plt.subplots(len(periods), len(polarisations), figsize=(14, 8), sharey=True, sharex=True)

for j, period in enumerate(periods):
    period_data = model_components[period]
    dump_indices = period_data['dump_indices']
    select = np.isin(dumps, dump_indices)
    on_mask = period_data['on_mask']

    flag_period = flags.squeeze[select, :, 0]
    atm_period = period_data['temperatures']['atmospheric'][:, :, 0] * 1000  # K
    synch_period = period_data['temperatures']['synchrotron'] * 100          # K

    for k, polarization in enumerate(polarisations):
        spillover = period_data['temperatures'][f'spillover_{polarization}'] * 1000  # K
        point_source = period_data['temperatures'][f'point_source_{polarization}']   # K

        components = {
            'point_source':      point_source,
            'atmosphere x1000':  atm_period,
            'spillover x1000':   spillover,
            'synchrotronx100': synch_period,
        }

        for label, component in components.items():
            masked = ma.masked_array(component, mask=flag_period)
            on_off = ma.mean(masked[on_mask], axis=0) - ma.mean(masked[~on_mask], axis=0)
            axes[j, k].plot(freq_MHz[freq_mask], on_off[freq_mask], label=label, alpha=0.7)

        axes[j, k].set_title(f'{period} — {period_data["calibrator"]} — {polarization}')
        axes[j, k].set_ylabel('on - off [K]')
        axes[j, k].legend(fontsize=8)
        axes[j, k].grid(True, alpha=0.3)
        if j == len(periods) - 1:
            axes[j, k].set_xlabel('Frequency [MHz]')

plt.tight_layout()
plt.show()


## Compare Primary Beam HH vs VV polarizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

i_dump = n_dumps // 2

# Beam gain
beam_HH = period_data['beam_gain_HH'][i_dump, :]
beam_VV = period_data['beam_gain_VV'][i_dump, :]

axes[0].plot(freq_MHz[freq_mask], beam_HH[freq_mask], 'r-', label='HH', linewidth=2)
axes[0].plot(freq_MHz[freq_mask], beam_VV[freq_mask], 'b--', label='VV', linewidth=2)
axes[0].set_xlabel('Frequency [MHz]')
axes[0].set_ylabel('Beam Gain')
axes[0].set_title('Primary Beam Gain (HH vs VV)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Ratio HH/VV
ratio = beam_HH / beam_VV
axes[1].plot(freq_MHz[freq_mask], ratio[freq_mask], 'k-', linewidth=2)
axes[1].axhline(y=1, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Frequency [MHz]')
axes[1].set_ylabel('Beam Gain Ratio (HH/VV)')
axes[1].set_title('Polarization Ratio')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Residuals per calibrator, per dish and polarisation using gain

In [ ]:
freq_mask = (freq_MHz >= 580.0) & (freq_MHz <= 1015.0)
periods = period_keys
n_periods = len(periods)
n_receivers = len(track_data.receivers)

dumps = np.array(track_data._dumps())
flags = track_data.flags.combine(threshold=1)

fig, axes = plt.subplots(n_receivers, n_periods, figsize=(6 * n_periods, 3 * n_receivers), sharex=True)

for j, period in enumerate(periods):
    period_data = model_components[period]
    dump_indices = period_data['dump_indices']
    select = np.isin(dumps, dump_indices)
    times = track_data.timestamps.squeeze[select]
    times_min = (times - times[0]) / 60.0

    for i_receiver in range(n_receivers):
        receiver = track_data.receivers[i_receiver]
        polarization = 'HH' if receiver.polarisation == 'h' else 'VV'

        vis_period = track_data.visibility.squeeze[select, :, i_receiver]
        flag_period = flags.squeeze[select, :, i_receiver]

        synch_period = period_data['temperatures']['synchrotron']
        atm_period = period_data['temperatures']['atmospheric'][:, :, i_receiver]
        rec_temp = period_data['temperatures']['receiver'][:, i_receiver]
        spillover = period_data['temperatures']['spillover_HH' if polarization == 'HH' else 'spillover_VV']
        point_source = period_data['temperatures']['point_source_HH' if polarization == 'HH' else 'point_source_VV']
        gain = period_data['gain'][:, i_receiver]  # (n_freq,)

        model_total = atm_period + point_source + rec_temp + spillover + synch_period
        vis_masked = ma.masked_array(vis_period, mask=flag_period)
        model_masked = ma.masked_array(model_total, mask=flag_period)

        vis_zeromean = vis_masked - ma.mean(vis_masked, axis=0)
        model_zeromean = model_masked - ma.mean(model_masked, axis=0)

        residuals = (vis_zeromean / gain - model_zeromean)*1000  # mK (n_dumps, n_freq)
#        residuals = model_zeromean*1000  # mK (n_dumps, n_freq)
#        residuals = vis_zeromean   # mK (n_dumps, n_freq)

        ax = axes[i_receiver, j]
        im = ax.pcolormesh(
            freq_MHz[freq_mask],
            times_min,
            residuals[:, freq_mask],
            shading='auto',
            cmap='RdBu_r'
        )
        plt.colorbar(im, ax=ax, label='Residual [mK]')
        ax.set_title(f'{period} — {receiver}', fontsize=8)
        ax.set_ylabel('Time [min]')
        if i_receiver == n_receivers - 1:
            ax.set_xlabel('Frequency [MHz]')

plt.suptitle('Residuals (calibrated - model)', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
freq_mask = (freq_MHz >= 580.0) & (freq_MHz <= 1015.0)
periods = period_keys
n_receivers = len(track_data.receivers)

dumps = np.array(track_data._dumps())
flags = track_data.flags.combine(threshold=1)

for period in periods:
    period_data = model_components[period]
    dump_indices = period_data['dump_indices']
    select = np.isin(dumps, dump_indices)
    times = track_data.timestamps.squeeze[select]
    times_min = (times - times[0]) / 60.0
    calibrator = period_data['calibrator']

    for polarization in ['HH', 'VV']:
        receivers_pol = [
            (i, r) for i, r in enumerate(track_data.receivers)
            if ('h' if polarization == 'HH' else 'v') == r.polarisation
        ]

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        title = f'{period} ({calibrator}) — {polarization}'

        for i_receiver, receiver in receivers_pol:
            vis_period = track_data.visibility.squeeze[select, :, i_receiver]
            flag_period = flags.squeeze[select, :, i_receiver]

            synch_period = period_data['temperatures']['synchrotron']
            atm_period = period_data['temperatures']['atmospheric'][:, :, i_receiver]
            rec_temp = period_data['temperatures']['receiver'][:, i_receiver]
            spillover = period_data['temperatures'][f'spillover_{polarization}']
            point_source = period_data['temperatures'][f'point_source_{polarization}']
            gain = period_data['gain'][:, i_receiver]  # (n_freq,)

            model_total = atm_period + point_source + rec_temp + spillover + synch_period
            vis_masked = ma.masked_array(vis_period, mask=flag_period)
            model_masked = ma.masked_array(model_total, mask=flag_period)

            vis_zeromean = vis_masked - ma.mean(vis_masked, axis=0)
            model_zeromean = model_masked - ma.mean(model_masked, axis=0)

            residuals = (vis_zeromean / gain - model_zeromean) * 1000  # mK

            # Median over time vs frequency
            axes[0].plot(freq_MHz[freq_mask], ma.median(residuals[:, freq_mask], axis=0), label=str(receiver), alpha=0.8)

            # Median over frequency vs time
            axes[1].plot(times_min, ma.median(residuals[:, freq_mask], axis=1), label=str(receiver), alpha=0.8)

        axes[0].set_xlabel('Frequency [MHz]')
        axes[0].set_ylabel('Residual [mK]')
        axes[0].set_title('Median over time')
        axes[0].axhline(0, color='k', linewidth=0.5, linestyle='--')
        axes[0].legend(fontsize=8)
        axes[0].grid(True, alpha=0.3)

        axes[1].set_xlabel('Time [min]')
        axes[1].set_ylabel('Residual [mK]')
        axes[1].set_title('Median over frequency')
        axes[1].axhline(0, color='k', linewidth=0.5, linestyle='--')
        axes[1].legend(fontsize=8)
        axes[1].grid(True, alpha=0.3)

        fig.suptitle(f'Residuals (calibrated - model) — {title}', fontsize=13)
        plt.tight_layout()
        plt.show()


## Median of residuals over dishes, per period and polarisation, using gain

In [ ]:
freq_mask = (freq_MHz >= 580.0) & (freq_MHz <= 1015.0)
periods = period_keys

dumps = np.array(track_data._dumps())
flags = track_data.flags.combine(threshold=1)

n_row_groups = len(periods)
height_ratios = [3, 1, 1] * n_row_groups
fig, axes = plt.subplots(
    3 * n_row_groups, 2,
    figsize=(12, (3 + 1 + 1) * n_row_groups),
    gridspec_kw={'height_ratios': height_ratios},
    sharex=False,
)

for j, period in enumerate(periods):
    period_data = model_components[period]
    dump_indices = period_data['dump_indices']
    select = np.isin(dumps, dump_indices)
    times = track_data.timestamps.squeeze[select]
    times_min = (times - times[0]) / 60.0

    on_mask = period_data['on_mask']
    transitions = np.where(np.diff(on_mask.astype(int)) != 0)[0]
    boundary_times = (times_min[transitions] + times_min[transitions + 1]) / 2

    residuals_HH, residuals_VV = [], []

    for i_receiver, receiver in enumerate(track_data.receivers):
        polarization = 'HH' if receiver.polarisation == 'h' else 'VV'

        vis_period = track_data.visibility.squeeze[select, :, i_receiver]
        flag_period = flags.squeeze[select, :, i_receiver]

        synch_period = period_data['temperatures']['synchrotron']
        atm_period = period_data['temperatures']['atmospheric'][:, :, i_receiver]
        rec_temp = period_data['temperatures']['receiver'][:, i_receiver]
        spillover = period_data['temperatures'][f'spillover_{polarization}']
        point_source = period_data['temperatures'][f'point_source_{polarization}']
        gain = period_data['gain'][:, i_receiver]

        model_total = atm_period + point_source + rec_temp + spillover + synch_period
        vis_masked = ma.masked_array(vis_period, mask=flag_period)
        model_masked = ma.masked_array(model_total, mask=flag_period)

        vis_zeromean = vis_masked - ma.mean(vis_masked, axis=0)
        model_zeromean = model_masked - ma.mean(model_masked, axis=0)

        residual = (vis_zeromean / gain - model_zeromean) * 1000  # mK
        (residuals_HH if polarization == 'HH' else residuals_VV).append(residual)

    for k, (pol, residuals_list) in enumerate([('HH', residuals_HH), ('VV', residuals_VV)]):
        residuals_median = ma.median(ma.stack(residuals_list, axis=0), axis=0)  # (n_dumps, n_freq)

        row_wf  = 3 * j      # waterfall
        row_sf  = 3 * j + 1  # spectrum (median over time)
        row_ts  = 3 * j + 2  # time series (median over freq)

        title = f'{period} ({period_data["calibrator"]}) — {pol}'

        # Waterfall
        ax_wf = axes[row_wf, k]
        im = ax_wf.pcolormesh(freq_MHz[freq_mask], times_min, residuals_median[:, freq_mask],
                               shading='auto', cmap='RdBu_r')
        plt.colorbar(im, ax=ax_wf, label='Residual [mK]')
        ax_wf.set_title(title, fontsize=9)
        ax_wf.set_ylabel('Time [min]')
        for bt in boundary_times:
            ax_wf.axhline(bt, color='black', linewidth=1, linestyle='--', alpha=0.7)

        # Median over time vs frequency
        ax_sf = axes[row_sf, k]
        ax_sf.plot(freq_MHz[freq_mask], ma.median(residuals_median[:, freq_mask], axis=0))
        ax_sf.axhline(0, color='k', linewidth=0.5, linestyle='--')
        ax_sf.set_ylabel('Residual [mK]')
        ax_sf.set_xlabel('Frequency [MHz]')
        ax_sf.grid(True, alpha=0.3)

        # Median over frequency vs time
        ax_ts = axes[row_ts, k]
        ax_ts.plot(times_min, ma.median(residuals_median[:, freq_mask], axis=1))
        ax_ts.axhline(0, color='k', linewidth=0.5, linestyle='--')
        for bt in boundary_times:
            ax_ts.axvline(bt, color='black', linewidth=1, linestyle='--', alpha=0.7)
        ax_ts.set_ylabel('Residual [mK]')
        ax_ts.set_xlabel('Time [min]')
        ax_ts.grid(True, alpha=0.3)

plt.suptitle('Median of residuals over dishes (calibrated - model)', y=1.01)
plt.tight_layout()
plt.show()


## Median of residuals over dishes, per period, for stokes I and pseudo-stokes Q, using gain. Model assumes unpolarised point source but HH and VV primary beams are different. 

In [ ]:
freq_mask = (freq_MHz >= 580.0) & (freq_MHz <= 1015.0)
periods = period_keys

dumps = np.array(track_data._dumps())
flags = track_data.flags.combine(threshold=1)

fig, axes = plt.subplots(len(periods) * 2, 2,
                         figsize=(12, 6 * len(periods)), sharex=False,
                         gridspec_kw={'height_ratios': [3, 1] * len(periods)})

for j, period in enumerate(periods):
    period_data = model_components[period]
    dump_indices = period_data['dump_indices']
    select = np.isin(dumps, dump_indices)
    times = track_data.timestamps.squeeze[select]
    times_min = (times - times[0]) / 60.0

    on_mask = period_data['on_mask']
    transitions = np.where(np.diff(on_mask.astype(int)) != 0)[0]
    boundary_times = (times_min[transitions] + times_min[transitions + 1]) / 2

    residuals_HH, residuals_VV = [], []

    for i_receiver, receiver in enumerate(track_data.receivers):
        polarization = 'HH' if receiver.polarisation == 'h' else 'VV'

        vis_period = track_data.visibility.squeeze[select, :, i_receiver]
        flag_period = flags.squeeze[select, :, i_receiver]

        synch_period = period_data['temperatures']['synchrotron']
        atm_period = period_data['temperatures']['atmospheric'][:, :, i_receiver]
        rec_temp = period_data['temperatures']['receiver'][:, i_receiver]
        spillover = period_data['temperatures'][f'spillover_{polarization}']
        point_source = period_data['temperatures'][f'point_source_{polarization}']
        gain = period_data['gain'][:, i_receiver]

        model_total = atm_period + point_source + rec_temp + spillover + synch_period
        vis_masked = ma.masked_array(vis_period, mask=flag_period)
        model_masked = ma.masked_array(model_total, mask=flag_period)

        vis_zeromean = vis_masked - ma.mean(vis_masked, axis=0)
        model_zeromean = model_masked - ma.mean(model_masked, axis=0)

        residual = (vis_zeromean / gain - model_zeromean) * 1000  # mK

        (residuals_HH if polarization == 'HH' else residuals_VV).append(residual)

    # Compute sum and difference per dish, then take median over dishes
    sum_list  = [(hh + vv)/2 for hh, vv in zip(residuals_HH, residuals_VV)]
    diff_list = [(hh - vv)/2 for hh, vv in zip(residuals_HH, residuals_VV)]

    for k, (label, residuals_list) in enumerate([('(HH+VV)/2', sum_list), ('(HH-VV)/2', diff_list)]):
        residuals_median = ma.median(ma.stack(residuals_list, axis=0), axis=0)

        ax_wf  = axes[j * 2,     k]  # waterfall
        ax_med = axes[j * 2 + 1, k]  # median over frequency

        im = ax_wf.pcolormesh(freq_MHz[freq_mask], times_min, residuals_median[:, freq_mask],
                              shading='nearest', cmap='RdBu_r')
        plt.colorbar(im, ax=ax_wf, label='Residual [mK]')
        ax_wf.set_title(f'{period} ({period_data["calibrator"]})— {label}', fontsize=9)
        ax_wf.set_ylabel('Time [min]')
        for bt in boundary_times:
            ax_wf.axhline(bt, color='black', linewidth=1, linestyle='--', alpha=0.7)

        freq_median = ma.median(residuals_median[:, freq_mask], axis=1)  # (n_dumps,)
        ax_med.plot(times_min, freq_median)
        ax_med.axhline(0, color='gray', linestyle='--', alpha=0.5)
        ax_med.set_ylabel('Residual [mK]')
        ax_med.set_xlabel('Time [min]')
        ax_med.grid(True, alpha=0.3)
        for bt in boundary_times:
            ax_med.axvline(bt, color='black', linewidth=1, linestyle='--', alpha=0.7)

plt.suptitle('Median of residuals over dishes (calibrated - model)', y=1.01)
plt.tight_layout()
plt.show()


## Parallatic angle calculations for Polarisation

In [ ]:
# katdal values (only for pointing centre)


import katdal

block_name = '1675021905'
data_folder = '/home/mgrsantos/projects/data/blocks'

rdb_path = f'{data_folder}/{block_name}/{block_name}/{block_name}_sdp_l0.full.rdb'
data = katdal.open(rdb_path)
antenna_names = [ant.name for ant in data.ants]
data.select(ants=antenna_names[0])

parangle_full = data.parangle  # (n_time_full, 1)

In [ ]:
def parallactic_angle(az, el, lat):
    """
    az, el : arrays in radians, shape (n_time,)
    lat    : scalar in radians
    returns: array of parallactic angles in degrees, shape (n_time,)
    """
    s = np.array([
        np.cos(el) * np.sin(az),
        np.cos(el) * np.cos(az),
        np.sin(el)
    ])  # (3, n_time)

    z = np.array([0.0, 0.0, 1.0])
    p = np.array([0.0, np.cos(lat), np.sin(lat)])

    zs = np.einsum('i,i...->...', z, s)  # (n_time,)
    ps = np.einsum('i,i...->...', p, s)  # (n_time,)

    vz = z[:, None] - zs * s  # (3, n_time)
    vp = p[:, None] - ps * s  # (3, n_time)

    cross = np.cross(vp, vz, axis=0)  # (3, n_time)
    num = np.einsum('i...,i...->...', s, cross)
    den = np.einsum('i...,i...->...', vp, vz)

    # katdal convention is negative of the calculation above
    # with the negative sign below, the equations are:
    # HH = I + Q·cos(2χ) - U·sin(2χ)
    # VV = I - Q·cos(2χ) + U·sin(2χ)

    return -np.rad2deg(np.arctan2(num, den))



In [ ]:

# Use first antenna (same for all dishes)
antenna0 = track_data.antennas[0]
lat = antenna0.ref_observer.lat  # radians

dumps = np.array(track_data._dumps())
times = track_data.timestamps.squeeze

fig, axes = plt.subplots(1, len(periods), figsize=(14, 4), sharey=True)

for j, period in enumerate(periods):
    dump_indices = model_components[period]['dump_indices']
    select = np.isin(dumps, dump_indices)
    times_min = (times[select] - times[select][0]) / 60.0

    az = np.deg2rad(track_data.azimuth.squeeze[select, 0])
    el = np.deg2rad(track_data.elevation.squeeze[select, 0])

    pa_computed = parallactic_angle(az, el, lat)
    pa_katdal   = parangle_full[dump_indices, 0]

    axes[j].plot(times_min, pa_computed - np.mean(pa_computed), label='computed')
    axes[j].plot(times_min, pa_katdal - np.mean(pa_katdal),   label='katdal', linestyle='--')
    axes[j].set_xlabel('Time [min]')
    axes[j].set_ylabel('Parallactic angle fluctuations [deg]')
    axes[j].set_title(f'{period} — {antenna_names[0]}')
    axes[j].legend()
    axes[j].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Expected variations in HH and VV in case the point source has Stokes Q (10% of I) - not including Stokes I contribution

In [ ]:
dumps = np.array(track_data._dumps())
times = track_data.timestamps.squeeze
freq_mask = (freq_MHz >= 580.0) & (freq_MHz <= 1015.0)

fig, axes = plt.subplots(len(periods), 2, figsize=(14, 6), sharey=True, sharex=False)

for j, period in enumerate(periods):
    period_data = model_components[period]
    dump_indices = period_data['dump_indices']
    select = np.isin(dumps, dump_indices)
    times_min = (times[select] - times[select][0]) / 60.0

    chi = np.deg2rad(parangle_full[dump_indices, 0])  # (n_dumps,)

    for k, polarization in enumerate(['HH', 'VV']):
        point_source = period_data['temperatures'][f'point_source_{polarization}']*1000  # mK (n_dumps, n_freq)

        sign = 1  if polarization == 'HH' else -1
        stokes_Q = sign*0.1*point_source[:, freq_mask] * np.cos(2 * chi)[:, None]  # 10% of stokes I (n_dumps, n_freq)
        stokes_Q = stokes_Q - np.ma.mean(stokes_Q, axis=0)
        stokes_Q_freq_median = np.ma.median(stokes_Q, axis=1)  # (n_dumps,)

        axes[j, k].plot(times_min, stokes_Q_freq_median)
        axes[j, k].set_title(f'{period} ({period_data["calibrator"]}) — {polarization}')
        axes[j, k].set_xlabel('Time [min]')
        axes[j, k].set_ylabel('Point_source - stokes Q [mK]')
        axes[j, k].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Access examples

Here are some examples of how to extract specific data from the structure.

In [ ]:
# Example 1: Get gain for specific period and receiver
period = 'before_scan'
i_receiver = 0
gain_recv = model_components[period]['gain'][:, i_receiver]  # (n_freq,)
print(f"Gain for period '{period}', receiver {i_receiver}: shape {gain_recv.shape}")

# Example 2: Get point source temperature for HH polarization
point_source_HH = model_components[period]['temperatures']['point_source_HH']  # (n_dumps, n_freq)
print(f"Point source HH: shape {point_source_HH.shape}")

# Example 3: Get atmospheric emission for all receivers at specific time
i_dump = 0
atm_time = model_components[period]['temperatures']['atmospheric'][i_dump, :, :]  # (n_freq, n_receivers)
print(f"Atmospheric at dump {i_dump}: shape {atm_time.shape}")

# Example 4: Compare gains between periods
periods = period_keys
if len(periods) >= 2:
    gain_before = model_components[periods[0]]['gain'][:, 0]
    gain_after = model_components[periods[1]]['gain'][:, 0]

    plt.figure(figsize=(10, 5))
    plt.plot(freq_MHz, gain_before, 'b-', label=f"{periods[0]}", linewidth=2)
    plt.plot(freq_MHz, gain_after, 'r--', label=f"{periods[1]}", linewidth=2)
    plt.xlabel('Frequency [MHz]')
    plt.ylabel('Gain')
    plt.title('Gain Comparison: Before vs After Scan (Receiver 0)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

## Summary

The `model_components` structure efficiently stores:

**Per period:**
- Metadata: calibrator name, dump indices
- Receiver-independent data (no duplication):
  - `beam_gain_HH`, `beam_gain_VV`: (n_dumps, n_freq)
  - `point_source_HH`, `point_source_VV`: (n_dumps, n_freq)
  - `spillover_HH`, `spillover_VV`: (n_dumps, n_freq)
- Receiver-specific data:
  - `gain`: (n_freq, n_receivers)
  - `atmospheric`: (n_dumps, n_freq, n_receivers)
  - `receiver`: (n_freq, n_receivers)

In [ ]:
import os

# Noise diode results are now per-period dicts from noise_diode_signal_plugin.
nd_pickle_path = os.path.join(os.path.dirname(pickle_path), 'noise_diode_signal_plugin.pickle')
with open(nd_pickle_path, 'rb') as f:
    _ctx = pickle.load(f)
nd_avg_dict = _ctx.get(ResultEnum.NOISE_DIODE_EXCESS_AVERAGE).result   # {period: (n_freq, n_recv)}
duty        = _ctx.get(ResultEnum.NOISE_DIODE_DUTY_CYCLE).result       # scalar
del _ctx
gc.collect()

freq_MHz  = track_data.frequencies.squeeze / 1.0e6
freq_mask = (freq_MHz >= 580.0) & (freq_MHz <= 1015.0)
pol_list  = [('HH', 'h'), ('VV', 'v')]

fig, axes = plt.subplots(len(period_keys), len(pol_list),
                         figsize=(7 * len(pol_list), 4 * len(period_keys)),
                         squeeze=False, sharex=True, sharey=True)

for i_period, period in enumerate(period_keys):
    nd_temp = nd_avg_dict[period]
    for i_pol, (pol, pol_char) in enumerate(pol_list):
        ax = axes[i_period, i_pol]
        for i_receiver, receiver in enumerate(track_data.receivers):
            if receiver.polarisation != pol_char:
                continue
            ax.plot(freq_MHz[freq_mask], nd_temp[freq_mask, i_receiver], alpha=0.8, label=str(receiver))
        ax.set_title(f"{period} ({model_components[period]['calibrator']}) — {pol}")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8, ncol=2)

for ax in axes[-1, :]:
    ax.set_xlabel('Frequency [MHz]')
for ax in axes[:, 0]:
    ax.set_ylabel('noise diode excess')
plt.tight_layout()
plt.show()


In [ ]:
import os

# Per-period excess average dict from noise_diode_signal_plugin.
nd_pickle_path = os.path.join(os.path.dirname(pickle_path), 'noise_diode_signal_plugin.pickle')
with open(nd_pickle_path, 'rb') as f:
    _ctx = pickle.load(f)
nd_avg_dict = _ctx.get(ResultEnum.NOISE_DIODE_EXCESS_AVERAGE).result   # {period: (n_freq, n_recv)}
del _ctx
gc.collect()

freq_MHz  = track_data.frequencies.squeeze / 1.0e6
freq_mask = (freq_MHz >= 580.0) & (freq_MHz <= 1015.0)

ratio = nd_avg_dict['after_scan'] / nd_avg_dict['before_scan']       # (n_freq, n_recv)

pol_list = [('HH', 'h'), ('VV', 'v')]
fig, axes = plt.subplots(1, len(pol_list), figsize=(7 * len(pol_list), 5), sharex=True, sharey=True)
for ax, (pol, pol_char) in zip(axes, pol_list):
    for i_receiver, receiver in enumerate(track_data.receivers):
        if receiver.polarisation != pol_char:
            continue
        ax.plot(freq_MHz[freq_mask], ratio[freq_mask, i_receiver], label=str(receiver), alpha=0.8)
    ax.axhline(1.0, color='k', lw=0.8, ls='--')
    ax.set_xlabel('Frequency [MHz]')
    ax.set_title(f'ND excess ratio  after / before — {pol}')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
axes[0].set_ylabel('after / before')
plt.tight_layout()
plt.show()


In [ ]:

# Per-period excess average dict from noise_diode_signal_plugin.
nd_pickle_path = os.path.join(os.path.dirname(pickle_path), 'noise_diode_signal_plugin.pickle')
with open(nd_pickle_path, 'rb') as f:
    _ctx = pickle.load(f)
nd_avg_dict = _ctx.get(ResultEnum.NOISE_DIODE_EXCESS_AVERAGE).result   # {period: (n_freq, n_recv)}
del _ctx
gc.collect()

freq_MHz  = track_data.frequencies.squeeze / 1.0e6
freq_mask = (freq_MHz >= 580.0) & (freq_MHz <= 1015.0)

ratio = nd_avg_dict['after_scan'] / nd_avg_dict['before_scan']       # (n_freq, n_recv)

percentage_fluctuations = 100 * (ratio - ma.median(ratio,axis=0)) / ma.median(ratio,axis=0)
pol_list = [('HH', 'h'), ('VV', 'v')]
fig, axes = plt.subplots(1, len(pol_list), figsize=(7 * len(pol_list), 5), sharex=True, sharey=True)
for ax, (pol, pol_char) in zip(axes, pol_list):
    for i_receiver, receiver in enumerate(track_data.receivers):
        if receiver.polarisation != pol_char:
            continue
        ax.plot(freq_MHz[freq_mask], percentage_fluctuations[freq_mask, i_receiver], label=str(receiver), alpha=0.8)
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_xlabel('Frequency [MHz]')
    ax.set_title(f'ND excess ratio (after/before) variation — {pol}')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
axes[0].set_ylabel('Percentage')
plt.tight_layout()
plt.show()


In [ ]:

method = 'boxcar'   # 'gaussian' or 'boxcar'
scale  = 41            # gaussian sigma in channels  (or boxcar window size if method='boxcar')

#method = 'gaussian'   # 'gaussian' or 'boxcar'
#scale  = 15            # gaussian sigma in channels  (or boxcar window size if method='boxcar')

before_s = smooth_freq(nd_avg_dict['before_scan'], scale, method)
after_s  = smooth_freq(nd_avg_dict['after_scan'],  scale, method)
ratio = after_s / before_s

percentage_fluctuations = 100 * (ratio - ma.median(ratio,axis=0)) / ma.median(ratio,axis=0)
pol_list = [('HH', 'h'), ('VV', 'v')]
fig, axes = plt.subplots(1, len(pol_list), figsize=(7 * len(pol_list), 5), sharex=True, sharey=True)
for ax, (pol, pol_char) in zip(axes, pol_list):
    for i_receiver, receiver in enumerate(track_data.receivers):
        if receiver.polarisation != pol_char:
            continue
        ax.plot(freq_MHz[freq_mask], percentage_fluctuations[freq_mask, i_receiver], label=str(receiver), alpha=0.8)
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_xlabel('Frequency [MHz]')
    ax.set_title(f'ND excess ratio (after/before) variation — {pol}')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
axes[0].set_ylabel('Percentage')
plt.tight_layout()
plt.show()

In [ ]:

method = 'gaussian'   # 'gaussian' or 'boxcar'
scale  = 15            # gaussian sigma in channels  (or boxcar window size if method='boxcar')

before_s = smooth_freq(nd_avg_dict['before_scan'], scale, method)
after_s  = smooth_freq(nd_avg_dict['after_scan'],  scale, method)
ratio = after_s / before_s

percentage_fluctuations = 100 * (ratio - ma.median(ratio,axis=0)) / ma.median(ratio,axis=0)
pol_list = [('HH', 'h'), ('VV', 'v')]
fig, axes = plt.subplots(1, len(pol_list), figsize=(7 * len(pol_list), 5), sharex=True, sharey=True)
for ax, (pol, pol_char) in zip(axes, pol_list):
    for i_receiver, receiver in enumerate(track_data.receivers):
        if receiver.polarisation != pol_char:
            continue
        ax.plot(freq_MHz[freq_mask], percentage_fluctuations[freq_mask, i_receiver], label=str(receiver), alpha=0.8)
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_xlabel('Frequency [MHz]')
    ax.set_title(f'ND excess ratio (after/before) variation — {pol}')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
axes[0].set_ylabel('Percentage')
plt.tight_layout()
plt.show()

In [ ]:
import os

# Noise diode results are now per-period dicts from noise_diode_signal_plugin.
nd_pickle_path = os.path.join(os.path.dirname(pickle_path), 'noise_diode_signal_plugin.pickle')
with open(nd_pickle_path, 'rb') as f:
    _ctx = pickle.load(f)
nd_avg_dict = _ctx.get(ResultEnum.NOISE_DIODE_EXCESS_AVERAGE).result   # {period: (n_freq, n_recv)}
duty        = _ctx.get(ResultEnum.NOISE_DIODE_DUTY_CYCLE).result       # scalar
del _ctx
gc.collect()

freq_MHz  = track_data.frequencies.squeeze / 1.0e6
freq_mask = (freq_MHz >= 580.0) & (freq_MHz <= 1015.0)
pol_list  = [('HH', 'h'), ('VV', 'v')]

fig, axes = plt.subplots(len(period_keys), len(pol_list),
                         figsize=(7 * len(pol_list), 4 * len(period_keys)),
                         squeeze=False, sharex=True, sharey=True)

for i_period, period in enumerate(period_keys):
    nd_temp = (nd_avg_dict[period] / model_components[period]['gain']) / duty   # [K]
    for i_pol, (pol, pol_char) in enumerate(pol_list):
        ax = axes[i_period, i_pol]
        for i_receiver, receiver in enumerate(track_data.receivers):
            if receiver.polarisation != pol_char:
                continue
            ax.plot(freq_MHz[freq_mask], nd_temp[freq_mask, i_receiver], alpha=0.8, label=str(receiver))
        ax.set_title(f"{period} ({model_components[period]['calibrator']}) — {pol}")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8, ncol=2)

for ax in axes[-1, :]:
    ax.set_xlabel('Frequency [MHz]')
for ax in axes[:, 0]:
    ax.set_ylabel('Calibrated noise diode [K]')
plt.tight_layout()
plt.show()


In [ ]:
import os

# Per-firing excess and firing times, per period, from noise_diode_signal_plugin.
nd_pickle_path = os.path.join(os.path.dirname(pickle_path), 'noise_diode_signal_plugin.pickle')
with open(nd_pickle_path, 'rb') as f:
    _ctx = pickle.load(f)
nd_excess          = _ctx.get(ResultEnum.NOISE_DIODE_EXCESS).result      # {period: (n_firings_p, n_freq, n_recv)}
noise_on_timestamp = _ctx.get(ResultEnum.NOISE_ON_TIMESTAMP).result      # {period: (n_firings_p,)}
del _ctx
gc.collect()

fig, axes = plt.subplots(1, len(period_keys), figsize=(7 * len(period_keys), 5),
                         squeeze=False, sharey=True)

for ax, period in zip(axes[0], period_keys):
    excess_p    = nd_excess[period]                    # (n_firings_p, n_freq, n_recv)
    t_p         = noise_on_timestamp[period]           # (n_firings_p,) sec since track start
    freq_median = ma.median(excess_p, axis=1)          # (n_firings_p, n_receivers); flagged channels excluded

    order = np.argsort(t_p)                            # time-ordered firings
    t_rel = (t_p[order] - t_p[order].min()) / 60.0     # min since this period's start

    for i_receiver, receiver in enumerate(track_data.receivers):
        ax.plot(t_rel, freq_median[order, i_receiver], 'o-', ms=3, lw=0.8, alpha=0.8, label=str(receiver))

    ax.set_title(f"{period} ({model_components[period]['calibrator']})")
    ax.set_xlabel('Time [min since period start]')
    ax.grid(alpha=0.3)

axes[0, 0].set_ylabel('ND excess (median over frequency)')
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
import os

# Per-firing excess and firing times, per period, from noise_diode_signal_plugin.
nd_pickle_path = os.path.join(os.path.dirname(pickle_path), 'noise_diode_signal_plugin.pickle')
with open(nd_pickle_path, 'rb') as f:
    _ctx = pickle.load(f)
nd_excess          = _ctx.get(ResultEnum.NOISE_DIODE_EXCESS).result      # {period: (n_firings_p, n_freq, n_recv)}
noise_on_timestamp = _ctx.get(ResultEnum.NOISE_ON_TIMESTAMP).result      # {period: (n_firings_p,)}
del _ctx
gc.collect()

pol_list = [('HH', 'h'), ('VV', 'v')]
fig, axes = plt.subplots(len(period_keys), len(pol_list),
                         figsize=(7 * len(pol_list), 4 * len(period_keys)),
                         squeeze=False, sharex='row', sharey=True)

for i_period, period in enumerate(period_keys):
    excess_p    = nd_excess[period]                    # (n_firings_p, n_freq, n_recv)
    t_p         = noise_on_timestamp[period]           # (n_firings_p,)
    freq_median = ma.median(excess_p, axis=1)          # (n_firings_p, n_receivers); flagged channels excluded

    order = np.argsort(t_p)                            # time-ordered firings
    t_rel = (t_p[order] - t_p[order].min()) / 60.0     # min since this period's start

    for i_pol, (pol, pol_char) in enumerate(pol_list):
        ax = axes[i_period, i_pol]
        for i_receiver, receiver in enumerate(track_data.receivers):
            if receiver.polarisation != pol_char:
                continue
            series = freq_median[order, i_receiver]    # this receiver, this period, time-ordered
            ref    = ma.mean(series)                   # time-mean reference
            pct    = 100.0 * (series - ref) / ref
            ax.plot(t_rel, pct, 'o-', ms=3, lw=0.8, alpha=0.8, label=str(receiver))
        ax.axhline(0, color='k', lw=0.5, alpha=0.5)
        ax.set_title(f"{period} ({model_components[period]['calibrator']}) — {pol}")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8, ncol=2)

for ax in axes[-1, :]:
    ax.set_xlabel('Time [min since period start]')
for ax in axes[:, 0]:
    ax.set_ylabel('ND excess deviation [%]')
plt.tight_layout()
plt.show()


In [ ]:
# gain / noise-diode excess average, per receiver, split before/after (rows) × HH/VV (cols).
if 'nd_avg_dict' not in globals():                      # load if not already in scope
    import os
    nd_pickle_path = os.path.join(os.path.dirname(pickle_path), 'noise_diode_signal_plugin.pickle')
    with open(nd_pickle_path, 'rb') as f:
        _ctx = pickle.load(f)
    nd_avg_dict = _ctx.get(ResultEnum.NOISE_DIODE_EXCESS_AVERAGE).result   # {period: (n_freq, n_recv)}
    del _ctx; gc.collect()

freq_MHz  = track_data.frequencies.squeeze / 1.0e6
freq_mask = (freq_MHz >= 580.0) & (freq_MHz <= 1015.0)
pol_list  = [('HH', 'h'), ('VV', 'v')]

fig, axes = plt.subplots(len(period_keys), len(pol_list),
                         figsize=(7 * len(pol_list), 4 * len(period_keys)),
                         squeeze=False, sharex=True, sharey=True)

for i_period, period in enumerate(period_keys):
    ratio = model_components[period]['gain'] / nd_avg_dict[period]   # (n_freq, n_recv) [counts/K / counts = 1/K]
    for i_pol, (pol, pol_char) in enumerate(pol_list):
        ax = axes[i_period, i_pol]
        for i_receiver, receiver in enumerate(track_data.receivers):
            if receiver.polarisation != pol_char:
                continue
            ax.plot(freq_MHz[freq_mask], ratio[freq_mask, i_receiver], alpha=0.8, label=str(receiver))
        ax.set_title(f"{period} ({model_components[period]['calibrator']}) — {pol}")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8, ncol=2)

for ax in axes[-1, :]:
    ax.set_xlabel('Frequency [MHz]')
for ax in axes[:, 0]:
    ax.set_ylabel('gain / ND excess')
plt.tight_layout()
plt.show()


In [ ]:
# % fluctuation along frequency of  (gain/nd_excess)_after / (gain/nd_excess)_before,
# per receiver, split HH / VV.
method = 'gaussian'   # 'gaussian' or 'boxcar'
scale  = 15           # gaussian sigma in channels (or boxcar window size if method='boxcar')

if 'nd_avg_dict' not in globals():
    import os
    nd_pickle_path = os.path.join(os.path.dirname(pickle_path), 'noise_diode_signal_plugin.pickle')
    with open(nd_pickle_path, 'rb') as f:
        _ctx = pickle.load(f)
    nd_avg_dict = _ctx.get(ResultEnum.NOISE_DIODE_EXCESS_AVERAGE).result
    del _ctx; gc.collect()

freq_MHz  = track_data.frequencies.squeeze / 1.0e6
freq_mask = (freq_MHz >= 580.0) & (freq_MHz <= 1015.0)
pol_list  = [('HH', 'h'), ('VV', 'v')]

# smooth each spectrum in frequency, then form (gain/nd)_after / (gain/nd)_before
g_after  = smooth_freq(model_components['after_scan']['gain'],  scale, method)
g_before = smooth_freq(model_components['before_scan']['gain'], scale, method)
nd_after  = smooth_freq(nd_avg_dict['after_scan'],  scale, method)
nd_before = smooth_freq(nd_avg_dict['before_scan'], scale, method)

ratio = (g_after / nd_after) / (g_before / nd_before)        # (n_freq, n_recv)

fig, axes = plt.subplots(1, len(pol_list), figsize=(7 * len(pol_list), 4),
                         squeeze=False, sharex=True, sharey=True)

for ax, (pol, pol_char) in zip(axes[0], pol_list):
    for i_receiver, receiver in enumerate(track_data.receivers):
        if receiver.polarisation != pol_char:
            continue
        spec = ratio[:, i_receiver]
        ref  = np.ma.median(spec[freq_mask])                 # band reference (frequency median)
        pct  = 100.0 * (spec - ref) / ref
        ax.plot(freq_MHz[freq_mask], pct[freq_mask], alpha=0.8, label=str(receiver))
    ax.axhline(0, color='k', lw=0.5, alpha=0.5)
    ax.set_title(f"(gain/nd)  after / before — {pol}")
    ax.set_xlabel('Frequency [MHz]')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, ncol=2)

axes[0, 0].set_ylabel('fluctuation [%]')
plt.tight_layout()
plt.show()



## Waterfall of the difference between two antennas, HH and VV, after gain calibration

In [ ]:
from museek.receiver import Receiver

freq_mask = (freq_MHz >= 580.0) & (freq_MHz <= 1015.0)
periods = period_keys

dumps = np.array(track_data._dumps())
flags = track_data.flags.combine(threshold=1)

antenna_pair = Receiver.receivers_to_antennas(track_data.receivers)[:2]
print(f'Comparing antennas: {antenna_pair}')

for period in periods:
    period_data = model_components[period]
    dump_indices = period_data['dump_indices']
    select = np.isin(dumps, dump_indices)
    times_period = track_data.timestamps.squeeze[select]
    times_min = (times_period - times_period[0]) / 60.0

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    for ax, (pol, pol_char) in zip(axes, [('HH', 'h'), ('VV', 'v')]):
        calibrated = {}
        for antenna_name in antenna_pair:
            i_receiver = next(
                i for i, r in enumerate(track_data.receivers)
                if r.antenna_name == antenna_name and r.polarisation == pol_char
            )
            vis_period = track_data.visibility.squeeze[select, :, i_receiver]
            flag_period = flags.squeeze[select, :, i_receiver]
            gain = period_data['gain'][:, i_receiver]  # (n_freq,)

            vis_masked = ma.masked_array(vis_period, mask=flag_period)
            calibrated[antenna_name] = vis_masked / gain

        diff = calibrated[antenna_pair[0]] - calibrated[antenna_pair[1]]

        im = ax.pcolormesh(
            freq_MHz[freq_mask],
            times_min,
            diff[:, freq_mask],
            shading='auto',
            cmap='RdBu_r'
        )
        plt.colorbar(im, ax=ax, label='Difference [K]')
        ax.set_xlabel('Frequency [MHz]')
        ax.set_title(f'{pol}: {antenna_pair[0]} - {antenna_pair[1]}')

    axes[0].set_ylabel('Time [min]')
    plt.suptitle(f'{period} ({period_data["calibrator"]}) — gain calibrated antenna difference')
    plt.tight_layout()
    plt.show()
